# 01 · Data: NBA player game logs

**Goal (Phase 1):** build one clean table, one row per player per game, from the raw logs in `nba/data/raw/`.

Raw files (downloaded by `nba/scripts/download_game_logs.py`, never edited):
- `player_gamelogs_<season>.csv`: one row per player per game **played** (2021-22 to 2025-26, regular season)
- `team_gamelogs_<season>.csv`: one row per team per game

Target columns: `player_id, player, team, opponent, game_id, date, home, minutes, points, fga, fg3a, fta, ...`

In [1]:
import pandas as pd
import numpy as np
from fbml.config import ROOT

raw = ROOT / "nba" / "data" / "raw"
files = sorted(raw.glob("player_gamelogs_*.csv"))

players = pd.concat(
    [pd.read_csv(f, dtype={"GAME_ID": str}) for f in files],
    ignore_index=True,
)
players.head()

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,FANTASY_PTS,VIDEO_AVAILABLE
0,22021,2546,Carmelo Anthony,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19,LAL vs. GSW,L,...,4,2,0,0,1,2,9,-5,15.8,1
1,22021,200765,Rajon Rondo,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19,LAL vs. GSW,L,...,0,5,2,0,3,2,3,1,13.5,1
2,22021,201566,Russell Westbrook,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19,LAL vs. GSW,L,...,5,4,0,0,4,4,8,-23,16.0,1
3,22021,201599,DeAndre Jordan,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19,LAL vs. GSW,L,...,2,1,1,0,0,3,2,2,8.9,1
4,22021,203076,Anthony Davis,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19,LAL vs. GSW,L,...,11,2,1,2,0,0,33,-2,58.2,1


In [2]:
print(players.shape)
print(players["SEASON_ID"].value_counts().sort_index())

(131291, 32)
SEASON_ID
22021    26039
22022    25894
22023    26401
22024    26306
22025    26651
Name: count, dtype: int64


In [3]:
players['date'] = pd.to_datetime(players['GAME_DATE'])
players['home'] = (~players['MATCHUP'].str.contains('@')).astype(int)
players['opponent'] = players['MATCHUP'].str.split(' ').str[-1]

In [4]:

players = players[players['MIN'] > 0]
players.info()

<class 'pandas.DataFrame'>
Index: 130830 entries, 0 to 131290
Data columns (total 35 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   SEASON_ID          130830 non-null  int64         
 1   PLAYER_ID          130830 non-null  int64         
 2   PLAYER_NAME        130830 non-null  str           
 3   TEAM_ID            130830 non-null  int64         
 4   TEAM_ABBREVIATION  130830 non-null  str           
 5   TEAM_NAME          130830 non-null  str           
 6   GAME_ID            130830 non-null  str           
 7   GAME_DATE          130830 non-null  str           
 8   MATCHUP            130830 non-null  str           
 9   WL                 130830 non-null  str           
 10  MIN                130830 non-null  int64         
 11  FGM                130830 non-null  int64         
 12  FGA                130830 non-null  int64         
 13  FG_PCT             124506 non-null  float64       
 14  FG3M

In [7]:
players = players.rename(
    columns={'SEASON_ID': 'season', 'PLAYER_ID': 'player_id', 'PLAYER_NAME': 'player', 'GAME_ID': 'game_id', 'MIN': 'minutes', 'TEAM_ABBREVIATION': 'team', 'PTS': 'points', 'AST': 'assists', 'REB': 'rebounds', 'STL': 'steals', 'BLK': 'blocks', 'TOV': 'turnovers', 'FGM': 'fgm', 'FGA': 'fga', 'FG3M': 'fg3m', 'FG3A': 'fg3a', 'FTM': 'ftm', 'FTA': 'fta', 'PLUS_MINUS': 'plus_minus'} )
players.sort_values(by=['date', 'player_id'], inplace=True)
players.reset_index(drop=True, inplace=True)

In [6]:
games = players[['season', 'player_id', 'player', 'team', 'opponent', 'game_id',
                 'date', 'home', 'minutes', 'points', 'fga', 'fg3a', 'fta']]

print(games.shape)
games.head()

(130830, 13)


,season,player_id,player,team,opponent,game_id,date,home,minutes,points,fga,fg3a,fta
0,22021,2544,LeBron James,LAL,GSW,0022100002,2021-10-19,1,37,34,23,11,6
1,22021,2546,Carmelo Anthony,LAL,GSW,0022100002,2021-10-19,1,26,9,9,4,2
2,22021,2730,Dwight Howard,LAL,GSW,0022100002,2021-10-19,1,13,5,2,0,4
3,22021,2738,Andre Iguodala,GSW,LAL,0022100002,2021-10-19,0,23,12,7,5,2
4,22021,200746,LaMarcus Aldridge,BKN,MIL,0022100001,2021-10-19,0,16,1,2,0,4


In [9]:
games.isnull().sum()

season       0
player_id    0
player       0
team         0
opponent     0
game_id      0
date         0
home         0
minutes      0
points       0
fga          0
fg3a         0
fta          0
dtype: int64

In [10]:
games.duplicated(subset = ['game_id', 'player_id']).sum()   


np.int64(0)

In [11]:
processed = ROOT / "nba" / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

games.to_parquet(processed / "games.parquet", index=False)
print("saved", games.shape, "->", processed / "games.parquet")

saved (130830, 13) -> C:\Users\Guninder Singh\Desktop\football-value-bets\nba\data\processed\games.parquet
